In [34]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv("../data/cleaned_products.csv")

# 1. RTX 5080 with 8 GB is impossible (title says 16 GB; the LLM read system RAM as VRAM)
bad = (df["gpu_model"] == "RTX 5080") & (df["vram_gb"] != 16)
print("Fixed RTX 5080 rows:", bad.sum())
df.loc[bad, "vram_gb"] = 16

# 2. GPU variant and numeric GPU features
df["gpu_variant"] = df["gpu_model"] + " " + df["vram_gb"].astype(int).astype(str) + "GB"

gpu = df["gpu_model"].str.extract(r"^(?:RTX|GTX)\s(\d{2})(\d{2})(\s+Ti)?$")
df["gpu_gen"] = pd.to_numeric(gpu[0])        # 30, 40, 50
df["gpu_tier"] = pd.to_numeric(gpu[1])       # 50, 60, 70, 80, 90
df["gpu_ti"] = gpu[2].notna().astype(int)    # 1 for Ti models

# 3. Target
df["log_price"] = np.log(df["Price"])


Fixed RTX 5080 rows: 1


In [35]:

# 4. CPU parsing (order matters: more specific patterns first)
CPU_PATTERNS = [
    ("Core Ultra", "Intel", r"ULTRA\s*([3579])\s*(\d{3})\s*([A-Z]*)"),
    ("Core i",     "Intel", r"\bI\s*([3579])\s*(\d{4,5})\s*([A-Z]*)"),
    ("Core",       "Intel", r"\bI\s*([3579])\s*(\d{3})(?!\d)\s*([A-Z]*)"),   # "I5 210H" is really Core 5 210H
    ("Core",       "Intel", r"\bCORE\s*([3579])\s*(\d{3})\s*([A-Z]*)"),
    ("Ryzen AI",   "AMD",   r"RYZEN\s*AI\s*([3579])\s*(\d{3})\s*([A-Z]*)"),
    ("Ryzen AI",   "AMD",   r"RYZEN\s*([3579])\s*AI\s*(\d{3})\s*([A-Z]*)"),  # "RYZEN 9 AI 365"
    ("Ryzen",      "AMD",   r"\bR(?:YZEN)?\s*([3579])\s*(\d{3,4})\s*([A-Z]*)"),
]

def parse_cpu(raw):
    empty = {"cpu_brand": None, "cpu_family": None, "cpu_tier": np.nan,
             "cpu_series": np.nan, "cpu_suffix": None}
    if pd.isna(raw):
        return pd.Series(empty)
    s = str(raw).upper().replace("-", " ")
    for family, brand, pattern in CPU_PATTERNS:
        m = re.search(pattern, s)
        if m:
            tier, number, suffix = m.groups()
            if "PLUS" in s:
                suffix += " PLUS"
            series = int(number[:2]) if family == "Core i" else int(number[0])
            return pd.Series({"cpu_brand": brand, "cpu_family": family,
                              "cpu_tier": int(tier), "cpu_series": series,
                              "cpu_suffix": suffix or "NONE"})
    return pd.Series(empty)

cpu_cols = ["cpu_brand", "cpu_family", "cpu_tier", "cpu_series", "cpu_suffix"]
cpu = df["cpu_model"].apply(parse_cpu)
df = pd.concat([df.drop(columns=cpu_cols, errors="ignore"), cpu], axis=1)
print("Unparsed CPUs:", df["cpu_family"].isna().sum())

# 5. Save
feature_cols = [
    "Link", "Name", "group_id", "Price", "log_price",
    "brand", "cpu_brand", "cpu_family", "cpu_tier", "cpu_series", "cpu_suffix",
    "ram_gb", "storage_gb", "screen_size", "refresh_rate_hz",
    "gpu_model", "gpu_variant", "gpu_gen", "gpu_tier", "gpu_ti", "vram_gb", "os",
]
df[feature_cols].to_csv("../data/features.csv", index=False, encoding="utf-8-sig")

saved = pd.read_csv("../data/features.csv")
print(saved.shape)
print(saved.isna().sum()[lambda s: s > 0].to_string())

Unparsed CPUs: 1
(1681, 22)
cpu_brand     1
cpu_family    1
cpu_tier      1
cpu_series    1
cpu_suffix    1
gpu_gen       3
gpu_tier      3


In [36]:
df["log_price"] = np.log(df["Price"])

gpu = df["gpu_model"].str.extract(r"^(?:RTX|GTX)\s(\d{2})(\d{2})(\s+Ti)?$")
df["gpu_gen"] = pd.to_numeric(gpu[0])        # 30, 40, 50
df["gpu_tier"] = pd.to_numeric(gpu[1])       # 50, 60, 70, 80, 90
df["gpu_ti"] = gpu[2].notna().astype(int)    # 1 for Ti models (RTX Pro 3000 stays NaN)

feature_cols = [
    "Link", "Name", "group_id", "Price", "log_price", "brand", "cpu_brand", "cpu_family", "cpu_tier", "cpu_series", "cpu_suffix",
    "ram_gb", "storage_gb", "screen_size", "refresh_rate_hz",
    "gpu_model", "gpu_variant", "gpu_gen", "gpu_tier", "gpu_ti", "vram_gb", "os",
]
df[feature_cols].to_csv("../data/features.csv", index=False, encoding="utf-8-sig")

print(df[feature_cols].shape)
print(df[feature_cols].isna().sum()[lambda s: s > 0].to_string())

(1681, 22)
cpu_brand     1
cpu_family    1
cpu_tier      1
cpu_series    1
cpu_suffix    1
gpu_gen       3
gpu_tier      3


In [37]:
print(pd.read_csv("../data/features.csv").columns.tolist())

['Link', 'Name', 'group_id', 'Price', 'log_price', 'brand', 'cpu_brand', 'cpu_family', 'cpu_tier', 'cpu_series', 'cpu_suffix', 'ram_gb', 'storage_gb', 'screen_size', 'refresh_rate_hz', 'gpu_model', 'gpu_variant', 'gpu_gen', 'gpu_tier', 'gpu_ti', 'vram_gb', 'os']


In [38]:
saved = pd.read_csv("../data/features.csv")
print(saved.shape)
print(saved.isna().sum()[lambda s: s > 0].to_string())

(1681, 22)
cpu_brand     1
cpu_family    1
cpu_tier      1
cpu_series    1
cpu_suffix    1
gpu_gen       3
gpu_tier      3


## Summary: what this notebook did

**Input:** `data/cleaned_products.csv` (1,681 listings)
**Output:** `data/features.csv` (1,681 rows x 22 columns): model-ready features plus the target.

### Steps

1. **Fixed one impossible value.** One RTX 5080 listing had 8 GB VRAM (the title says 16 GB; the LLM read the 8 GB of system RAM as VRAM). It was set to 16 GB.
2. **GPU variant.** `gpu_variant` combines the GPU name and VRAM (e.g. `RTX 5070 12GB`), because the same GPU name exists in two real variants with very different prices.
3. **GPU numeric features.** `gpu_gen` (30/40/50), `gpu_tier` (50/60/70/80/90) and `gpu_ti` (0/1) were parsed from the GPU name so that linear models can use the ordering. `RTX Pro 3000` has no generation/tier (3 rows left as missing).
4. **CPU parsing.** 138 messy CPU strings were reduced to five columns with regex rules:

| Column | Meaning | Example |
|---|---|---|
| `cpu_brand` | Manufacturer | Intel, AMD |
| `cpu_family` | Product family | Core Ultra, Core i, Core, Ryzen, Ryzen AI |
| `cpu_tier` | Level inside the family | 3, 5, 7, 9 |
| `cpu_series` | Series number (comparable only within a family) | 13 (13620H), 2 (Ultra 275HX) |
| `cpu_suffix` | Power class | HX, H, HS, HX PLUS, NONE (no suffix) |

   Two data quirks were handled: `I5 210H` is really Core 5 210H, and `RYZEN 9 AI 365` is Ryzen AI 9 365. One listing (`AMD RYZEN 7`, no model number) cannot be parsed and stays missing.
5. **Target.** `log_price = log(Price)`, because the raw price is right-skewed (skewness 1.52 vs 0.29 after the log).
6. **Brand.** Kept as `brand`. Rare brands (fewer than 20 listings) will be grouped inside the modeling pipeline (`OneHotEncoder(min_frequency=20)`), so the counts come from training data only.

### Column roles

- **Target:** `log_price` (and `Price` for reporting in TL).
- **Features:** brand, CPU columns, RAM, storage, screen size, refresh rate, GPU columns, VRAM, OS.
- **Not model inputs:** `Link` and `Name` (traceability only) and `group_id`, which is used for a grouped train/test split so that the same product from different sellers never lands on both sides.

### Deliberately not done here

No encoding, scaling, or imputation. These steps belong inside a scikit-learn `Pipeline` in Phase 3 so that nothing is learned from the test data.

### Limitations

- Missing VRAM for RTX 5070 and RTX 3050 was filled with the per-GPU mode earlier, and both GPUs exist in two real variants, so `gpu_variant` can be wrong for some filled rows.
- The CPU rules were written for the strings in this dataset; new listings may need new rules.
- LLM extraction errors that were not caught during cleaning may remain (accuracy against a hand-labeled sample is still to be measured).